In [1]:
from verify_objects import *
from verify_relation import *
from verifier import *

---
# Example1 : Gradient descent
$$ x^{k+1} = x^k - \frac{1}{L} \nabla f(x^k) $$
---

## Define materials

In [2]:
x = Sequence('x') # seqeunce generated by gradient descent
f = Function('f')
g = Operator('g') # Grdaient f
k = IterationCount('k')
L = Scalar('L')   # smoothness parameter
A = ScalarSequence('A')

In [3]:
x_star = x('\star')
f_star = f(x_star)

### Define Lyapunov function
$$ \Phi^n = A^n ( f(x^n) - f^\star ) + \frac{L}{2} \left\| x^n - x^\star \right\|^2 $$

In [4]:
def Lypunov(n):
    return A(n) * ( f(x(n)) - f_star )+ L/2 * NS( x(n)-x_star )

## The 'proof' we want to show is
$$
\begin{align*}
\Phi^{k+1} - \Phi^{k}
&= \lambda_1 \left( f(x^k) - f^\star + \left\langle \nabla f(x^k),\, x^\star - x^k \right\rangle \right) \\
&\quad + \lambda_2 \left( f(x^{k+1}) - f(x^k) + \left\langle \nabla f(x^k),\, x^{k+1} - x^k \right\rangle + \frac{L}{2}\left\|x^k - x^{k+1}\right\|^2 \right) \\
&\quad - \frac{1}{2L}\left\|\nabla f(x^k)\right\|^2
\end{align*}
$$
### where
$$
\begin{align*}
\lambda_1 &= A^{k+1} - A^k, \qquad
\lambda_2 = A^{k+1}
\end{align*}
$$

In [5]:
lambda_1 = A(k+1) - A(k)
lambda_2 = A(k+1)

In [6]:
ineq1 = f(x(k)) - f_star + IP(g(x(k)), x_star-x(k))
ineq2 = f(x(k+1)) - (f(x(k)) + IP(g(x(k)),x(k+1)-x(k)) + L/2 * NS(x(k)-x(k+1))  )

In [7]:
LHS = Lypunov(k+1) - Lypunov(k)
RHS = lambda_1 * ineq1 + lambda_2 * ineq2 - 0.5/L * A(k) * NS(g(x(k)))

In [8]:
( LHS - RHS ).simplify()

( -0.5 L+0.5 A_{1+k} L ) || x^{k} ||^2+( -A_{1+k}+A_{k} ) < g(x^{k}), x^{\star} >+( 0.5 A_{1+k} L+0.5 L ) || x^{1+k} ||^2-A_{1+k} L < x^{1+k}, x^{k} >-A_{k} < g(x^{k}), x^{k} >-L < x^{1+k}, x^{\star} >+0.5 A_{k} L^{-1} || g(x^{k}) ||^2+A_{1+k} < g(x^{k}), x^{1+k} >+L < x^{\star}, x^{k} >

## Substituting Example

In [9]:
def x_def(n) :
    statement = Equal(x(n+1), x(n)-1/L*g(x(n)))
    statement.truth = 'Axiom'
    return statement

def A_def(n):
    statement = Equal(A(n+1), A(n)+1)
    statement.truth = 'Axiom'
    return statement

In [10]:
( LHS - RHS ).simplify()

( -0.5 L+0.5 A_{1+k} L ) || x^{k} ||^2+( -A_{1+k}+A_{k} ) < g(x^{k}), x^{\star} >+( 0.5 A_{1+k} L+0.5 L ) || x^{1+k} ||^2-A_{1+k} L < x^{1+k}, x^{k} >-A_{k} < g(x^{k}), x^{k} >-L < x^{1+k}, x^{\star} >+0.5 A_{k} L^{-1} || g(x^{k}) ||^2+A_{1+k} < g(x^{k}), x^{1+k} >+L < x^{\star}, x^{k} >

In [11]:
( LHS - RHS ).substitute(x_def(k)).simplify()

( -0.5 A_{1+k} L^{-1}+0.5 A_{k} L^{-1}+0.5 L^{-1} ) || g(x^{k}) ||^2+( -1.0-A_{k}+A_{1+k} ) < g(x^{k}), x^{k} >+( 1-A_{1+k}+A_{k} ) < g(x^{k}), x^{\star} >

In [12]:
( LHS - RHS ).substitute(x_def(k)).substitute(A_def(k)).simplify()

0